# Preliminary

In [ ]:
!git clone https://github.com/sht037-lgtm/Q-Vtree.git

In [ ]:
# log in HuggingFace
!pip install -U huggingface_hub
!hf auth login --token "input your hf token here"

!pip install qwen-vl-utils[decord]==0.0.8

In [ ]:
# download Qwen2.5-VL-3B-Instruct Model
%cd Q-Vtree/checkpoints
!python download.py
!ls

In [ ]:
%cd /kaggle/working/Q-Vtree

import json
import os
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from transformers import AutoProcessor

model_path = "checkpoints/Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(model_path)

# load tree model (Global-Compact variant)
from qwen.global_compact import Qwen2_5_VLForConditionalGenerationWithTree

tree_model = Qwen2_5_VLForConditionalGenerationWithTree.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="eager",
)
tree_model.eval()

print("tree model (Global-Compact) loaded")

# Demo

In [ ]:
demo_img = Image.open("img/demo_6.jpeg").convert("RGB")
demo_question = "What breed is the dog in the image?"

In [ ]:
# baseline inference
pred_base = tree_model.infer(processor, demo_img, demo_question, use_tree=False)
print("Baseline:", pred_base)

In [ ]:
# Global-Compact tree inference
pred_tree = tree_model.infer(processor, demo_img, demo_question, use_tree=True)
print("Global-Compact tree:", pred_tree)

n_sel = sum(tree_model._debug_num_selected_tokens)
n_tot = sum(tree_model._debug_num_total_tokens)
print(f"Selected {n_sel} / {n_tot} patches ({n_sel/n_tot:.1%})")

In [ ]:
# visualize: selected patch overlay
patch_ids = tree_model._debug_patch_ids[0]
patch_scores_raw = tree_model._debug_patch_scores[0]

from qwen_vl_utils import process_vision_info
messages = [{"role": "user", "content": [
    {"type": "image", "image": demo_img},
    {"type": "text",  "text": demo_question},
]}]
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                   padding=True, return_tensors="pt")

grid_h = inputs["image_grid_thw"][0][1].item() // 2
grid_w = inputs["image_grid_thw"][0][2].item() // 2
patch_size = 28

img_vis = demo_img.convert("RGB").resize((grid_w * patch_size, grid_h * patch_size))
overlay = Image.new("RGBA", img_vis.size, (0, 0, 0, 0))
draw = ImageDraw.Draw(overlay)

for idx in patch_ids.tolist():
    r, c = idx // grid_w, idx % grid_w
    x0, y0 = c * patch_size, r * patch_size
    draw.rectangle([x0, y0, x0 + patch_size, y0 + patch_size],
                   fill=(255, 0, 0, 80), outline=(255, 0, 0, 255))

out = Image.alpha_composite(img_vis.convert("RGBA"), overlay)
plt.figure(figsize=(8, 8 * grid_h / grid_w))
plt.imshow(out)
plt.axis("off")
plt.title("Selected patches")
plt.show()

In [ ]:
# visualize: attention score heatmap
scores = patch_scores_raw.float().detach().cpu()
scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-6)
scores_map = F.interpolate(
    scores.reshape(1, 1, grid_h, grid_w),
    size=(demo_img.size[1], demo_img.size[0]),
    mode="bicubic", align_corners=False,
).squeeze().numpy()

plt.figure(figsize=(10, 10 * demo_img.size[1] / demo_img.size[0]))
plt.imshow(demo_img)
plt.imshow(scores_map, cmap="jet", alpha=0.5)
plt.axis("off")
plt.title("Patch attention score heatmap")
plt.show()

In [ ]:
# visualize: compact image
compact_img = tree_model._debug_compact_image
print(f"Original: {demo_img.size}  ->  Compact: {compact_img.size}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(demo_img); axes[0].axis("off"); axes[0].set_title("Original")
axes[1].imshow(compact_img); axes[1].axis("off"); axes[1].set_title("Compact (LPD)")
plt.tight_layout()
plt.show()

# Evaluation

In [ ]:
# download evaluation datasets (vstar, hrbench, mmbench, mme_realworld)
%cd datasets
!python download.py
!ls
%cd ..

from qwen.evaluate import (
    run_vstar_inference,
    evaluate_vstar_predictions,
    run_hrbench_inference,
    evaluate_hrbench_predictions,
    run_mmbench_inference,
    evaluate_mmbench_predictions,
    run_mme_realworld_inference,
    evaluate_mme_realworld_predictions,
)

## V-Star

In [ ]:
# base qwen + vstar
from transformers import Qwen2_5_VLForConditionalGeneration
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_path, torch_dtype=torch.float16, device_map="auto"
)
base_model.eval()

pred_file = run_vstar_inference(
    model=base_model,
    processor=processor,
    model_type="base_qwen",
    run_name="base_qwen_vstar",
)
evaluate_vstar_predictions(pred_file)

In [ ]:
# tree qwen (Global-Compact) + vstar
pred_file = run_vstar_inference(
    model=tree_model,
    processor=processor,
    model_type="tree_qwen",
    run_name="tree_qwen_vstar_global_compact",
)
evaluate_vstar_predictions(pred_file)

## HR-Bench

In [ ]:
# base qwen + hrbench 4k
pred_file = run_hrbench_inference(
    model=base_model,
    processor=processor,
    split="4k",
    model_type="base_qwen",
    run_name="base_qwen_hrbench4k",
)
evaluate_hrbench_predictions(pred_file)

In [ ]:
# tree qwen (Global-Compact) + hrbench 4k
pred_file = run_hrbench_inference(
    model=tree_model,
    processor=processor,
    split="4k",
    model_type="tree_qwen",
    run_name="tree_qwen_hrbench4k_global_compact",
)
evaluate_hrbench_predictions(pred_file)

In [ ]:
# base qwen + hrbench 8k
pred_file = run_hrbench_inference(
    model=base_model,
    processor=processor,
    split="8k",
    model_type="base_qwen",
    run_name="base_qwen_hrbench8k",
)
evaluate_hrbench_predictions(pred_file)

In [ ]:
# tree qwen (Global-Compact) + hrbench 8k
pred_file = run_hrbench_inference(
    model=tree_model,
    processor=processor,
    split="8k",
    model_type="tree_qwen",
    run_name="tree_qwen_hrbench8k_global_compact",
)
evaluate_hrbench_predictions(pred_file)

## MMBench

In [ ]:
# base qwen + mmbench
pred_file = run_mmbench_inference(
    model=base_model,
    processor=processor,
    model_type="base_qwen",
    run_name="base_qwen_mmbench",
    resume=True,
)
evaluate_mmbench_predictions(pred_file)

In [ ]:
# tree qwen (Global-Compact) + mmbench
pred_file = run_mmbench_inference(
    model=tree_model,
    processor=processor,
    model_type="tree_qwen",
    run_name="tree_qwen_mmbench_global_compact",
    resume=True,
)
evaluate_mmbench_predictions(pred_file)

## MME-RealWorld

In [ ]:
# base qwen + mme_realworld
pred_file = run_mme_realworld_inference(
    model=base_model,
    processor=processor,
    model_type="base_qwen",
    run_name="base_qwen_mme_realworld",
    resume=True,
)
evaluate_mme_realworld_predictions(pred_file)

In [ ]:
# tree qwen (Global-Compact) + mme_realworld
pred_file = run_mme_realworld_inference(
    model=tree_model,
    processor=processor,
    model_type="tree_qwen",
    run_name="tree_qwen_mme_realworld_global_compact",
    resume=True,
)
evaluate_mme_realworld_predictions(pred_file)